# Momentum Factor Strategy — Interactive Notebook

This notebook walks through the complete 12-1 month momentum backtest step by step.

**Strategy**: Buy top-decile stocks ranked by 12-1 month price return, rebalance monthly.

---

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['figure.facecolor'] = '#0d1117'
matplotlib.rcParams['axes.facecolor'] = '#161b22'
matplotlib.rcParams['text.color'] = '#e6edf3'

print('Environment ready.')

## Step 1 — Generate Market Data

In [ ]:
from data_generator import generate_price_data

prices, benchmark, sector_map = generate_price_data(seed=42)

print(f'Universe    : {prices.shape[1]} stocks')
print(f'Date range  : {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'Trading days: {len(prices):,}')
print(f'\nSector breakdown:')
print(sector_map.value_counts().to_string())

In [ ]:
# Plot a sample of stock price paths
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Price paths (normalised)
sample_tickers = prices.columns[:10]
norm_prices = prices[sample_tickers] / prices[sample_tickers].iloc[0]
for ticker in sample_tickers:
    axes[0].plot(norm_prices.index, norm_prices[ticker], lw=0.7, alpha=0.7)
axes[0].plot(benchmark.index, benchmark['SPY'] / benchmark['SPY'].iloc[0],
             color='#d29922', lw=1.5, ls='--', label='Benchmark')
axes[0].set_title('Normalised Price Paths (10 Sample Stocks)', color='#e6edf3')
axes[0].legend()

# Sector distribution
counts = sector_map.value_counts()
axes[1].barh(counts.index, counts.values, color='#58a6ff', alpha=0.8)
axes[1].set_title('Stocks per Sector', color='#e6edf3')
for spine in axes[1].spines.values():
    spine.set_color('#21262d')

plt.tight_layout()
plt.show()

## Step 2 — Signal Construction

In [ ]:
from signals import construct_signals

signals = construct_signals(prices)
mom_raw  = signals['momentum_raw']
mom_rank = signals['momentum_rank']

print('Signal coverage (fraction of universe with valid signal):')
print(signals['signal_quality'].describe())

# Show cross-sectional signal distribution on a sample date
sample_date = mom_raw.dropna(how='all').index[100]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(mom_raw.loc[sample_date].dropna() * 100, bins=20,
             color='#58a6ff', alpha=0.8, edgecolor='#0d1117')
axes[0].set_title(f'Raw Momentum Signal — {sample_date.date()}', color='#e6edf3')
axes[0].set_xlabel('12-1m Return (%)', color='#8b949e')

axes[1].hist(mom_rank.loc[sample_date].dropna(), bins=20,
             color='#3fb950', alpha=0.8, edgecolor='#0d1117')
axes[1].set_title('Percentile Rank', color='#e6edf3')
axes[1].set_xlabel('Rank [0, 1]', color='#8b949e')

for ax in axes:
    ax.tick_params(colors='#8b949e')
    for sp in ax.spines.values(): sp.set_color('#21262d')
plt.tight_layout()
plt.show()

## Step 3 — Run Backtest

In [ ]:
from portfolio import run_backtest

result = run_backtest(
    prices=prices,
    momentum_rank=mom_rank,
    top_pct=0.10,
    long_only=True,
    tc_bps=10,
)

strat_returns = result['returns']
bench_returns = benchmark['SPY'].pct_change().reindex(strat_returns.index, fill_value=0)

print(f'Strategy return series: {len(strat_returns):,} days')
print(f'Total rebalances      : {len(result["signal_dates"])}')
print(f'Avg monthly turnover  : {result["turnover_history"].mean()*100:.1f}%')

## Step 4 — Performance Metrics

In [ ]:
from metrics import compute_full_metrics

IS_END   = '2018-12-31'
OOS_START = '2019-01-01'

for label, s, e in [
    ('Full Period (2008-2023)', strat_returns.index[0], strat_returns.index[-1]),
    ('In-Sample (2008-2018)',   strat_returns.index[0], pd.Timestamp(IS_END)),
    ('Out-of-Sample (2019-2023)', pd.Timestamp(OOS_START), strat_returns.index[-1]),
]:
    print(f'\n{'='*55}')
    print(f'  {label}')
    print('='*55)
    m = compute_full_metrics(strat_returns.loc[s:e], bench_returns.loc[s:e])
    print(m.to_string())

## Step 5 — Generate Tearsheet

In [ ]:
from tearsheet import generate_tearsheet

generate_tearsheet(
    strat_returns=strat_returns,
    bench_returns=bench_returns,
    turnover_history=result['turnover_history'],
    strat_name='Momentum Long-Only (12-1m)',
    bench_name='SPY Benchmark',
    split_date=OOS_START,
    output_path='../results/tearsheet_notebook.png',
)

from IPython.display import Image
Image('../results/tearsheet_notebook.png', width=1200)